# Time Series Foundation Models — Benchmark probabilístico + Optuna

Notebook **auto-contido** que compara *time series foundation models* (TSFM) open-source
em **forecasting probabilístico**, com ajuste de hiperparâmetros de inferência via **Optuna**.

**Modelos comparados (zero-shot por padrão):**

| # | Modelo | HF id | ~Params | Covariáveis |
|---|--------|-------|---------|-------------|
| 1 | Chronos-Bolt Base | `amazon/chronos-bolt-base` | ~205M | não (nativo) |
| 2 | Chronos-2 | `amazon/chronos-2` | ~120M | **sim** (past + future) |
| 3 | TimesFM 2.0 | `google/timesfm-2.0-500m-pytorch` | ~500M | não |
| 4 | Moirai-2 small | `Salesforce/moirai-2.0-R-small` | ~small | dynamic real feats |
| — | AutoARIMA (baseline) | `statsforecast` | — | não |

> **Nota sobre TimesFM:** a tarefa menciona "TimesFM 2.5", mas apenas o checkpoint
> `google/timesfm-2.0-500m-pytorch` tem suporte limpo via `transformers`
> (`TimesFmModelForPrediction`). Por isso usamos o **2.0-500m** — robusto e cabe em 12 GB.

> **Fine-tuning (LoRA / Optuna Study B):** *fora de escopo nesta versão* (decisão do usuário).
> O Estudo A (tuning de inferência zero-shot) está totalmente implementado.

---

## Dependências (NÃO instalar dentro do notebook)

Instale num ambiente Python 3.11+ **antes** de abrir o notebook. Versões testadas/compatíveis:

```bash
# PyTorch com CUDA (ajuste o índice CUDA à sua máquina; RTX 3060 → cu121 funciona bem)
pip install "torch>=2.2,<2.6" --index-url https://download.pytorch.org/whl/cu121

# Foundation models
pip install "chronos-forecasting>=1.5.0"      # Chronos-Bolt + Chronos-2 (Chronos2Pipeline)
pip install "transformers>=4.48,<4.57" "accelerate>=0.30" "peft>=0.11"  # TimesFM 2.0
pip install "uni2ts>=1.2.0"                    # Moirai-2 (Moirai2Forecast/Moirai2Module) + gluonts

# Tuning + baseline + utilidades
pip install "optuna>=3.6" "statsforecast>=1.7" "plotly>=5.0"
pip install "pandas>=2.0" "numpy>=1.26,<2.2" "matplotlib>=3.7" "scipy>=1.11"
```

> ⚠️ **Conflitos de versão:** `uni2ts` costuma fixar uma faixa de `transformers` mais antiga
> que a exigida pelo TimesFM. Se houver conflito, instale `uni2ts` num ambiente separado **ou**
> deixe o notebook pular automaticamente o modelo cuja lib não carregar (os wrappers já tratam
> `ImportError` com aviso, sem derrubar o notebook).


## 1. Setup e diagnóstico

Detecção de GPU/VRAM, seed global reprodutível, helper de memória e **constantes globais**.
Tudo roda com *mixed precision* (`bf16`/`fp16`) quando há CUDA; caso contrário, *fallback* para CPU.


In [ ]:
from __future__ import annotations

import gc
import math
import random
import time
import warnings
from typing import Any, Callable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    _HAS_TORCH = True
except Exception as exc:  # pragma: no cover
    torch = None  # type: ignore
    _HAS_TORCH = False
    warnings.warn(f"PyTorch não disponível: {exc}. Modelos neurais serão pulados.")

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="gluonts")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)


def detect_device_dtype() -> tuple[str, "Any"]:
    """Escolhe device e dtype de mixed precision adequados ao hardware."""
    if _HAS_TORCH and torch.cuda.is_available():
        device = "cuda"
        # bf16 é preferível quando suportado (Ampere+, ex.: RTX 3060 suporta).
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    else:
        device = "cpu"
        dtype = torch.float32 if _HAS_TORCH else None
    return device, dtype


def diagnose() -> None:
    """Imprime diagnóstico de ambiente (CUDA, VRAM, versões)."""
    print(f"PyTorch disponível: {_HAS_TORCH}")
    if not _HAS_TORCH:
        print("  -> rodando sem torch; apenas AutoARIMA estará disponível.")
        return
    print(f"torch: {torch.__version__}")
    cuda_ok = torch.cuda.is_available()
    print(f"CUDA disponível: {cuda_ok}")
    if cuda_ok:
        idx = torch.cuda.current_device()
        props = torch.cuda.get_device_properties(idx)
        print(f"  GPU: {props.name}")
        print(f"  VRAM total: {props.total_memory / 1024**3:.2f} GB")
        print(f"  CUDA runtime: {torch.version.cuda}")
        print(f"  bf16 suportado: {torch.cuda.is_bf16_supported()}")
    else:
        print("  -> fallback para CPU (mais lento).")


DEVICE, DTYPE = detect_device_dtype()
diagnose()
print(f"\nDevice selecionado: {DEVICE} | dtype: {DTYPE}")


In [ ]:
# ---- Reprodutibilidade ----------------------------------------------------
SEED: int = 42


def set_global_seed(seed: int = SEED) -> None:
    """Fixa todas as fontes de aleatoriedade para reprodutibilidade (tol. 1e-4)."""
    random.seed(seed)
    np.random.seed(seed)
    if _HAS_TORCH:
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        # determinismo razoável sem matar a performance
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


set_global_seed(SEED)


# ---- Telemetria de memória ------------------------------------------------
def log_gpu_mem(tag: str = "") -> float:
    """Imprime e retorna o pico de VRAM (MB) desde o último reset. 0.0 se CPU."""
    if not (_HAS_TORCH and torch.cuda.is_available()):
        return 0.0
    torch.cuda.synchronize()
    alloc = torch.cuda.memory_allocated() / 1024**2
    peak = torch.cuda.max_memory_allocated() / 1024**2
    print(f"[VRAM] {tag:<28s} alloc={alloc:8.1f} MB | peak={peak:8.1f} MB")
    return peak


def reset_gpu_peak() -> None:
    """Zera o contador de pico de VRAM (chamar antes de medir um modelo)."""
    if _HAS_TORCH and torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()


def free_memory() -> None:
    """Coleta lixo e esvazia o cache CUDA (entre modelos, evita OOM em 12 GB)."""
    gc.collect()
    if _HAS_TORCH and torch.cuda.is_available():
        torch.cuda.empty_cache()


# ---- Constantes globais ---------------------------------------------------
HORIZON: int = 96            # H — passos a prever
CONTEXT_LENGTH: int = 512    # L — janela de contexto inicial (default)
N_TRIALS_INFERENCE: int = 30  # trials Optuna (Estudo A) por modelo
N_TRIALS_LORA: int = 20      # reservado; Estudo B (LoRA) fora de escopo nesta versão
QUANTILES: list[float] = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# IDs dos modelos (Hugging Face)
MODEL_IDS: dict[str, str] = {
    "chronos-bolt-base": "amazon/chronos-bolt-base",
    "chronos-2": "amazon/chronos-2",
    "timesfm-2.0": "google/timesfm-2.0-500m-pytorch",
    "moirai-2-small": "Salesforce/moirai-2.0-R-small",
}

print(f"HORIZON={HORIZON}  CONTEXT_LENGTH={CONTEXT_LENGTH}  SEED={SEED}")
print(f"N_TRIALS_INFERENCE={N_TRIALS_INFERENCE}  QUANTILES={QUANTILES}")


## 2. Carregamento e validação dos dados

Os dados **já existem** em formato Nixtla (`unique_id`, `ds`, `y`), opcionalmente com
covariáveis extras. O notebook **não baixa nem gera dados** — você fornece o `DataFrame`.

Utilitários implementados:

- `load_user_data` — valida e padroniza o painel, com relatório de diagnóstico.
- `summarize_panel` — perfil por série (comprimento, média, std, ACF lag-1, tendência via Mann-Kendall).
- `split_panel` — split temporal (treino / validação / teste) respeitando `unique_id`.
- `make_windows` — janelas deslizantes para avaliação.


In [ ]:
REQUIRED_COLS: tuple[str, ...] = ("unique_id", "ds", "y")


def _detect_freq(g: pd.DataFrame) -> str | None:
    """Tenta inferir a frequência de uma série ordenada por ds."""
    if len(g) < 3:
        return None
    try:
        return pd.infer_freq(g["ds"])
    except Exception:
        return None


def load_user_data(df: pd.DataFrame, freq: str = "H") -> pd.DataFrame:
    """Valida e padroniza um DataFrame no formato Nixtla.

    - Verifica colunas obrigatórias: unique_id, ds, y.
    - Garante ds como datetime e y como float.
    - Ordena por (unique_id, ds) e remove duplicatas exatas de (unique_id, ds).
    - Relata: nº de séries, comprimento por série, frequência detectada,
      missing values e gaps temporais.

    Args:
        df: DataFrame de entrada do usuário.
        freq: frequência pandas esperada (ex.: "H", "D", "15min"). Usada como
            referência para detectar gaps.

    Returns:
        DataFrame validado e ordenado.

    Raises:
        TypeError: se `df` não for um pandas.DataFrame.
        ValueError: se faltarem colunas obrigatórias ou os tipos forem inconversíveis.
    """
    if not isinstance(df, pd.DataFrame):
        raise TypeError(
            "Esperado um pandas.DataFrame no formato Nixtla. "
            "Você substituiu o placeholder `df = ...` pelo seu carregamento? "
            f"Recebido: {type(df)!r}."
        )

    missing = [c for c in REQUIRED_COLS if c not in df.columns]
    if missing:
        raise ValueError(
            f"Colunas obrigatórias ausentes: {missing}. "
            f"Formato Nixtla exige {list(REQUIRED_COLS)}. "
            f"Colunas encontradas: {list(df.columns)}."
        )

    out = df.copy()

    # ds -> datetime
    try:
        out["ds"] = pd.to_datetime(out["ds"])
    except Exception as exc:
        raise ValueError(f"Não foi possível converter 'ds' para datetime: {exc}") from exc

    # y -> float
    try:
        out["y"] = out["y"].astype(float)
    except Exception as exc:
        raise ValueError(f"Não foi possível converter 'y' para float: {exc}") from exc

    out["unique_id"] = out["unique_id"].astype(str)

    n_dup = out.duplicated(subset=["unique_id", "ds"]).sum()
    if n_dup:
        warnings.warn(f"{n_dup} linhas duplicadas (unique_id, ds) removidas.")
        out = out.drop_duplicates(subset=["unique_id", "ds"])

    out = out.sort_values(["unique_id", "ds"]).reset_index(drop=True)

    # ----- Relatório de diagnóstico -----
    covariates = [c for c in out.columns if c not in REQUIRED_COLS]
    lengths = out.groupby("unique_id").size()
    n_missing_y = int(out["y"].isna().sum())
    freqs = {uid: _detect_freq(g) for uid, g in out.groupby("unique_id")}
    detected = pd.Series(list(freqs.values())).value_counts(dropna=False)

    # gaps: contagem de séries cujo span/freq não bate com o nº de pontos
    n_gappy = 0
    for uid, g in out.groupby("unique_id"):
        try:
            full = pd.date_range(g["ds"].iloc[0], g["ds"].iloc[-1], freq=freq)
            if len(full) != len(g):
                n_gappy += 1
        except Exception:
            pass

    print("=" * 64)
    print("RELATÓRIO load_user_data")
    print("=" * 64)
    print(f"Séries (unique_id):      {out['unique_id'].nunique()}")
    print(f"Linhas totais:           {len(out)}")
    print(f"Comprimento por série:   min={lengths.min()} | mediana={int(lengths.median())} | max={lengths.max()}")
    print(f"Período (ds):            {out['ds'].min()}  ->  {out['ds'].max()}")
    print(f"Frequência esperada:     {freq}")
    print(f"Frequência detectada:    {dict(detected)}")
    print(f"Missing em y:            {n_missing_y}")
    print(f"Séries com gaps (vs {freq}): {n_gappy}")
    print(f"Covariáveis extras:      {covariates if covariates else 'nenhuma'}")
    print("=" * 64)

    if lengths.min() < HORIZON:
        warnings.warn(
            f"Há séries mais curtas que HORIZON={HORIZON}. "
            "O split/avaliação pode pular essas séries."
        )
    return out


In [ ]:
# ---- Estatística de perfil por série -------------------------------------
def _acf_lag1(y: np.ndarray) -> float:
    """Autocorrelação no lag 1 (NaN se série < 3 ou variância nula)."""
    y = np.asarray(y, dtype=float)
    y = y[~np.isnan(y)]
    if len(y) < 3 or np.std(y) == 0:
        return float("nan")
    a, b = y[:-1], y[1:]
    denom = np.std(a) * np.std(b)
    if denom == 0:
        return float("nan")
    return float(np.mean((a - a.mean()) * (b - b.mean())) / denom)


def _mann_kendall(y: np.ndarray) -> tuple[float, str]:
    """Teste de tendência de Mann-Kendall simplificado.

    Retorna (tau aproximado, rótulo) em {'crescente','decrescente','sem tendência'}.
    Usa estatística S normalizada; sem correção de empates (versão simplificada).
    """
    y = np.asarray(y, dtype=float)
    y = y[~np.isnan(y)]
    n = len(y)
    if n < 8:
        return float("nan"), "indeterminado"
    s = 0
    for k in range(n - 1):
        s += np.sign(y[k + 1:] - y[k]).sum()
    var_s = n * (n - 1) * (2 * n + 5) / 18.0
    if var_s == 0:
        return 0.0, "sem tendência"
    z = (s - np.sign(s)) / math.sqrt(var_s)
    tau = s / (0.5 * n * (n - 1))
    if abs(z) < 1.96:  # ~95%
        label = "sem tendência"
    else:
        label = "crescente" if z > 0 else "decrescente"
    return float(tau), label


def summarize_panel(df: pd.DataFrame) -> pd.DataFrame:
    """Estatísticas por série, úteis para interpretar os resultados depois.

    Colunas: length, mean, std, cv, acf_lag1, mk_tau, trend.
    """
    rows: list[dict[str, Any]] = []
    for uid, g in df.groupby("unique_id"):
        y = g["y"].to_numpy(dtype=float)
        mean = float(np.nanmean(y)) if len(y) else float("nan")
        std = float(np.nanstd(y)) if len(y) else float("nan")
        tau, trend = _mann_kendall(y)
        rows.append(
            {
                "unique_id": uid,
                "length": int(len(y)),
                "mean": mean,
                "std": std,
                "cv": float(std / mean) if mean not in (0.0, float("nan")) else float("nan"),
                "acf_lag1": _acf_lag1(y),
                "mk_tau": tau,
                "trend": trend,
            }
        )
    return pd.DataFrame(rows).set_index("unique_id")


In [ ]:
# ---- Split temporal e janelas --------------------------------------------
def split_panel(
    df: pd.DataFrame, h_test: int = HORIZON, h_val: int = HORIZON
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Split temporal por janela final, respeitando cada unique_id.

    Para cada série: os últimos `h_test` pontos viram teste, os `h_val`
    anteriores viram validação, e o restante é treino/histórico.
    Séries curtas demais (< h_test + h_val + 1) são descartadas com aviso.

    Returns:
        (train, val, test) — cada um no formato Nixtla.
    """
    train_parts, val_parts, test_parts, dropped = [], [], [], []
    for uid, g in df.groupby("unique_id"):
        g = g.sort_values("ds")
        n = len(g)
        if n < h_test + h_val + 1:
            dropped.append(uid)
            continue
        test_parts.append(g.iloc[n - h_test:])
        val_parts.append(g.iloc[n - h_test - h_val: n - h_test])
        train_parts.append(g.iloc[: n - h_test - h_val])
    if dropped:
        warnings.warn(
            f"{len(dropped)} série(s) curtas descartadas no split "
            f"(precisam de >= {h_test + h_val + 1} pontos): {dropped[:5]}"
            + (" ..." if len(dropped) > 5 else "")
        )
    empty = df.iloc[0:0]
    train = pd.concat(train_parts) if train_parts else empty
    val = pd.concat(val_parts) if val_parts else empty
    test = pd.concat(test_parts) if test_parts else empty
    return (
        train.reset_index(drop=True),
        val.reset_index(drop=True),
        test.reset_index(drop=True),
    )


def make_windows(
    df: pd.DataFrame, context_length: int = CONTEXT_LENGTH, horizon: int = HORIZON, stride: int | None = None
) -> list[dict[str, Any]]:
    """Gera janelas deslizantes (context, target) por série para avaliação.

    Cada janela é um dict: {unique_id, ds_target, context (np), target (np), df_context}.
    `context` é truncado adaptativamente se a série for menor que context_length.
    """
    if stride is None:
        stride = horizon
    windows: list[dict[str, Any]] = []
    for uid, g in df.groupby("unique_id"):
        g = g.sort_values("ds").reset_index(drop=True)
        y = g["y"].to_numpy(dtype=float)
        n = len(y)
        if n < horizon + 1:
            continue
        # primeira posição de início do target
        start = max(context_length, 1)
        for t in range(start, n - horizon + 1, stride):
            ctx = y[max(0, t - context_length): t]
            tgt = y[t: t + horizon]
            if len(tgt) < horizon:
                break
            windows.append(
                {
                    "unique_id": uid,
                    "ds_target": g["ds"].iloc[t: t + horizon].to_numpy(),
                    "context": ctx,
                    "target": tgt,
                    "df_context": g.iloc[max(0, t - context_length): t],
                }
            )
    return windows


### Célula de carregamento do usuário

**Substitua o placeholder abaixo** pelo seu carregamento de dados. Nada mais precisa mudar.


In [ ]:
# >>> SUBSTITUA AQUI PELO SEU CARREGAMENTO <<<
# Exemplo:
#   df = pd.read_parquet("meu_dataset.parquet")
#   df = pd.read_csv("meu_dataset.csv", parse_dates=["ds"])
df = ...  # placeholder — troque por um DataFrame Nixtla (unique_id, ds, y[, covariáveis])

DATA_FREQ: str = "H"  # ajuste à frequência dos seus dados (ex.: "H", "D", "15min")
df = load_user_data(df, freq=DATA_FREQ)
df.head()


In [ ]:
# Perfil das séries (interpretação posterior dos resultados)
panel_profile = summarize_panel(df)
panel_profile


In [ ]:
# Grid de plots (até 12 séries; amostra se houver mais)
def plot_panel_grid(df: pd.DataFrame, max_series: int = 12, seed: int = SEED) -> None:
    """Plota até `max_series` séries em grid; amostra reprodutível se houver mais."""
    uids = list(df["unique_id"].unique())
    if len(uids) > max_series:
        rng = np.random.default_rng(seed)
        uids = sorted(rng.choice(uids, size=max_series, replace=False).tolist())
    n = len(uids)
    ncols = min(3, n)
    nrows = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 2.6 * nrows), squeeze=False)
    for ax, uid in zip(axes.flat, uids):
        g = df[df["unique_id"] == uid].sort_values("ds")
        ax.plot(g["ds"], g["y"], lw=0.8)
        ax.set_title(str(uid), fontsize=9)
        ax.tick_params(labelsize=7)
    for ax in axes.flat[n:]:
        ax.axis("off")
    fig.suptitle("Painel de séries", fontsize=12)
    fig.tight_layout()
    plt.show()


plot_panel_grid(df)


## 3. Métricas

Implementadas inline: **MAE, RMSE, sMAPE, MASE** (ponto) e **WQL** (Weighted Quantile Loss)
e **CRPS** (aproximado pela média das *pinball losses* sobre a grade de quantis) para a
parte probabilística. Uma única função `evaluate` consolida tudo.


In [ ]:
def mae(y: np.ndarray, yhat: np.ndarray) -> float:
    return float(np.mean(np.abs(y - yhat)))


def rmse(y: np.ndarray, yhat: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y - yhat) ** 2)))


def smape(y: np.ndarray, yhat: np.ndarray) -> float:
    """sMAPE em % (denominador com epsilon para evitar divisão por zero)."""
    denom = (np.abs(y) + np.abs(yhat))
    denom = np.where(denom == 0, 1e-8, denom)
    return float(100.0 * np.mean(2.0 * np.abs(y - yhat) / denom))


def mase(y: np.ndarray, yhat: np.ndarray, train_history: np.ndarray, seasonality: int = 1) -> float:
    """Mean Absolute Scaled Error. Escala = MAE do naive sazonal no histórico."""
    h = np.asarray(train_history, dtype=float)
    h = h[~np.isnan(h)]
    m = seasonality if seasonality < len(h) else 1
    if len(h) <= m:
        return float("nan")
    scale = np.mean(np.abs(h[m:] - h[:-m]))
    if scale == 0:
        return float("nan")
    return float(np.mean(np.abs(y - yhat)) / scale)


def quantile_loss(y: np.ndarray, q_pred: np.ndarray, q: float) -> float:
    """Pinball loss (não normalizada) para um quantil q."""
    diff = y - q_pred
    return float(np.mean(np.maximum(q * diff, (q - 1.0) * diff)))


def wql(y: np.ndarray, quantiles_pred: np.ndarray, quantile_levels: list[float]) -> float:
    """Weighted Quantile Loss: soma das pinball losses normalizada por sum(|y|).

    quantiles_pred: shape (n_q, H). quantile_levels: list de tamanho n_q.
    """
    total = 0.0
    for i, q in enumerate(quantile_levels):
        diff = y - quantiles_pred[i]
        total += np.sum(np.maximum(q * diff, (q - 1.0) * diff))
    denom = np.sum(np.abs(y))
    denom = denom if denom != 0 else 1e-8
    return float(2.0 * total / (len(quantile_levels) * denom))


def crps_from_quantiles(y: np.ndarray, quantiles_pred: np.ndarray, quantile_levels: list[float]) -> float:
    """Aproximação do CRPS pela média (sobre a grade) de 2*pinball loss.

    Para uma grade uniforme de quantis, a média de 2*QL aproxima o CRPS.
    """
    vals = [2.0 * quantile_loss(y, quantiles_pred[i], q) for i, q in enumerate(quantile_levels)]
    return float(np.mean(vals))


def evaluate(
    forecast_dict: dict[str, np.ndarray],
    truth: np.ndarray,
    train_history: np.ndarray | None = None,
    quantile_levels: list[float] | None = None,
    seasonality: int = 1,
) -> dict[str, float]:
    """Calcula todas as métricas para uma previsão.

    Args:
        forecast_dict: deve conter 'median' (H,) e 'quantiles' (n_q, H).
        truth: valores reais (H,).
        train_history: histórico para escala do MASE (opcional).
        quantile_levels: níveis correspondentes a forecast_dict['quantiles'].
        seasonality: período sazonal para o naive do MASE.
    """
    if quantile_levels is None:
        quantile_levels = QUANTILES
    y = np.asarray(truth, dtype=float)
    median = np.asarray(forecast_dict["median"], dtype=float)
    q_pred = np.asarray(forecast_dict["quantiles"], dtype=float)

    out: dict[str, float] = {
        "MAE": mae(y, median),
        "RMSE": rmse(y, median),
        "sMAPE": smape(y, median),
        "WQL": wql(y, q_pred, quantile_levels),
        "CRPS": crps_from_quantiles(y, q_pred, quantile_levels),
    }
    out["MASE"] = (
        mase(y, median, train_history, seasonality)
        if train_history is not None
        else float("nan")
    )
    return out


## 4. Wrapper unificado

`TSFMWrapper` define a interface comum. Cada modelo tem uma subclasse que trata as
diferenças de API e devolve sempre `{'median': (H,), 'quantiles': (n_q, H)}`.

Princípios:

- **Carregar → usar → descarregar**: cada modelo é instanciado, usado e liberado
  (`unload()` faz `del` + `empty_cache()`) antes do próximo, para caber em 12 GB.
- **Truncamento adaptativo**: contexto maior que a série é truncado, com aviso — nunca quebra.
- **Covariáveis**: apenas Chronos-2 as usa nativamente; os demais avisam e ignoram.
- **Robustez**: falha de import/carregamento de um modelo é capturada e o modelo é pulado.


In [ ]:
def _to_quantile_grid(
    src_q: np.ndarray, src_levels: list[float], dst_levels: list[float]
) -> np.ndarray:
    """Reinterpola previsões de quantis de uma grade para outra (por passo de horizonte).

    src_q: (n_src, H). Retorna (n_dst, H).
    """
    src_levels = list(src_levels)
    H = src_q.shape[1]
    out = np.empty((len(dst_levels), H), dtype=float)
    for t in range(H):
        out[:, t] = np.interp(dst_levels, src_levels, src_q[:, t])
    return out


class TSFMWrapper:
    """Interface comum para os foundation models de séries temporais."""

    name: str = "base"
    supports_covariates: bool = False

    def __init__(self, model_id: str, device: str = DEVICE, dtype: "Any" = DTYPE) -> None:
        self.model_id = model_id
        self.device = device
        self.dtype = dtype
        self._model: Any = None
        self._n_params: int = 0
        self.load()

    # ---- ciclo de vida ----
    def load(self) -> None:  # pragma: no cover - sobrescrito
        raise NotImplementedError

    def unload(self) -> None:
        """Libera o modelo da memória (del + empty_cache)."""
        self._model = None
        free_memory()

    # ---- util ----
    def _fit_context(self, context: np.ndarray, max_len: int) -> np.ndarray:
        """Trunca o contexto adaptativamente quando a série é curta (com aviso)."""
        context = np.asarray(context, dtype=float)
        if len(context) > max_len:
            return context[-max_len:]
        if len(context) < max_len:
            warnings.warn(
                f"[{self.name}] série curta: contexto={len(context)} < L={max_len}; "
                "usando todo o histórico disponível.",
                stacklevel=2,
            )
        return context

    @property
    def num_params(self) -> int:
        return self._n_params

    def _autocast(self):
        """Context manager de mixed precision (no-op em CPU)."""
        if _HAS_TORCH and self.device == "cuda":
            return torch.autocast(device_type="cuda", dtype=self.dtype)
        import contextlib

        return contextlib.nullcontext()

    def predict(
        self,
        context: np.ndarray,
        h: int = HORIZON,
        num_samples: int = 100,
        quantiles: list[float] | None = None,
        covariates_past: np.ndarray | None = None,
        covariates_future: np.ndarray | None = None,
        **kwargs: Any,
    ) -> dict[str, np.ndarray]:  # pragma: no cover - sobrescrito
        raise NotImplementedError


In [ ]:
class ChronosBoltWrapper(TSFMWrapper):
    """Chronos-Bolt — modelo de quantis (não amostra). amazon/chronos-bolt-base."""

    name = "chronos-bolt-base"
    supports_covariates = False

    def load(self) -> None:
        from chronos import BaseChronosPipeline

        self._model = BaseChronosPipeline.from_pretrained(
            self.model_id,
            device_map=self.device,
            torch_dtype=self.dtype,
        )
        inner = getattr(self._model, "model", None)
        if inner is not None and hasattr(inner, "parameters"):
            self._n_params = sum(p.numel() for p in inner.parameters())

    def predict(
        self,
        context: np.ndarray,
        h: int = HORIZON,
        num_samples: int = 100,
        quantiles: list[float] | None = None,
        covariates_past: np.ndarray | None = None,
        covariates_future: np.ndarray | None = None,
        context_length: int | None = None,
        **kwargs: Any,
    ) -> dict[str, np.ndarray]:
        if covariates_past is not None or covariates_future is not None:
            warnings.warn(f"[{self.name}] não usa covariáveis; ignorando.")
        quantiles = quantiles or QUANTILES
        ctx = self._fit_context(context, context_length or CONTEXT_LENGTH)
        ctx_t = torch.tensor(ctx, dtype=torch.float32)
        with self._autocast():
            q_tensor, _mean = self._model.predict_quantiles(
                context=ctx_t,
                prediction_length=h,
                quantile_levels=quantiles,
            )
        # q_tensor: (batch=1, H, n_q) -> (n_q, H)
        q_np = q_tensor[0].float().cpu().numpy().T
        median_idx = quantiles.index(0.5) if 0.5 in quantiles else len(quantiles) // 2
        return {"median": q_np[median_idx], "quantiles": q_np}


In [ ]:
class Chronos2Wrapper(TSFMWrapper):
    """Chronos-2 — suporta covariáveis (past + future). amazon/chronos-2."""

    name = "chronos-2"
    supports_covariates = True

    def load(self) -> None:
        try:
            from chronos import Chronos2Pipeline
        except ImportError:
            from chronos import BaseChronosPipeline as Chronos2Pipeline  # fallback de nome
        self._model = Chronos2Pipeline.from_pretrained(
            self.model_id,
            device_map=self.device,
            torch_dtype=self.dtype,
        )
        inner = getattr(self._model, "model", None)
        if inner is not None and hasattr(inner, "parameters"):
            self._n_params = sum(p.numel() for p in inner.parameters())

    def predict(
        self,
        context: np.ndarray,
        h: int = HORIZON,
        num_samples: int = 100,
        quantiles: list[float] | None = None,
        covariates_past: np.ndarray | None = None,
        covariates_future: np.ndarray | None = None,
        context_length: int | None = None,
        **kwargs: Any,
    ) -> dict[str, np.ndarray]:
        quantiles = quantiles or QUANTILES
        ctx = self._fit_context(context, context_length or CONTEXT_LENGTH)

        # Chronos-2 trabalha com DataFrames/dicts; montamos um payload mínimo.
        # API pode variar entre versões — tentamos o caminho de quantis e caímos
        # para um caminho alternativo se necessário.
        target_df = pd.DataFrame({"target": ctx})
        try:
            with self._autocast():
                pred = self._model.predict_quantiles(
                    context=torch.tensor(ctx, dtype=torch.float32),
                    prediction_length=h,
                    quantile_levels=quantiles,
                )
            if isinstance(pred, tuple):
                q_tensor = pred[0]
            else:
                q_tensor = pred
            q_np = np.asarray(q_tensor)[0].T if np.asarray(q_tensor).ndim == 3 else np.asarray(q_tensor).T
        except Exception as exc:
            warnings.warn(f"[{self.name}] caminho predict_quantiles falhou ({exc}); usando predict genérico.")
            with self._autocast():
                out = self._model.predict([target_df], prediction_length=h)
            # esperado: lista de DataFrames com colunas de quantis
            fdf = out[0] if isinstance(out, (list, tuple)) else out
            cols = [c for c in fdf.columns if isinstance(c, (int, float))]
            q_np = _to_quantile_grid(
                fdf[sorted(cols)].to_numpy().T, sorted(cols), quantiles
            ) if cols else np.tile(fdf.iloc[:, 0].to_numpy(), (len(quantiles), 1))

        median_idx = quantiles.index(0.5) if 0.5 in quantiles else len(quantiles) // 2
        return {"median": q_np[median_idx], "quantiles": q_np}


In [ ]:
class TimesFMWrapper(TSFMWrapper):
    """TimesFM 2.0 via transformers. google/timesfm-2.0-500m-pytorch."""

    name = "timesfm-2.0"
    supports_covariates = False
    # quantis nativos do TimesFM (full_predictions inclui mean + 9 quantis 0.1..0.9)
    _native_levels = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

    def load(self) -> None:
        from transformers import TimesFmModelForPrediction

        self._model = TimesFmModelForPrediction.from_pretrained(
            self.model_id,
            torch_dtype=self.dtype,
        ).to(self.device)
        self._model.eval()
        self._n_params = sum(p.numel() for p in self._model.parameters())

    def predict(
        self,
        context: np.ndarray,
        h: int = HORIZON,
        num_samples: int = 100,
        quantiles: list[float] | None = None,
        covariates_past: np.ndarray | None = None,
        covariates_future: np.ndarray | None = None,
        context_length: int | None = None,
        freq: int = 0,
        **kwargs: Any,
    ) -> dict[str, np.ndarray]:
        if covariates_past is not None or covariates_future is not None:
            warnings.warn(f"[{self.name}] não usa covariáveis; ignorando.")
        quantiles = quantiles or QUANTILES
        # TimesFM 2.0 suporta contexto até 2048
        ctx = self._fit_context(context, min(context_length or CONTEXT_LENGTH, 2048))
        past = [torch.tensor(ctx, dtype=torch.float32)]
        freq_t = torch.tensor([freq], dtype=torch.long)
        with torch.no_grad(), self._autocast():
            out = self._model(past_values=past, freq=freq_t)
        mean_pred = np.asarray(out.mean_predictions[0].float().cpu())[:h]
        full = getattr(out, "full_predictions", None)
        if full is not None:
            full_np = np.asarray(full[0].float().cpu())  # (H, n_outputs)
            full_np = full_np[:h]
            n_out = full_np.shape[-1]
            # convenção comum: coluna 0 = mean, colunas 1.. = quantis 0.1..0.9
            if n_out >= len(self._native_levels) + 1:
                q_src = full_np[:, 1: 1 + len(self._native_levels)].T  # (n_q, H)
                q_np = _to_quantile_grid(q_src, self._native_levels, quantiles)
            else:
                q_np = np.tile(mean_pred, (len(quantiles), 1))
        else:
            q_np = np.tile(mean_pred, (len(quantiles), 1))
        median_idx = quantiles.index(0.5) if 0.5 in quantiles else len(quantiles) // 2
        return {"median": q_np[median_idx], "quantiles": q_np}


In [ ]:
class Moirai2Wrapper(TSFMWrapper):
    """Moirai-2 small via uni2ts. Salesforce/moirai-2.0-R-small (GluonTS)."""

    name = "moirai-2-small"
    supports_covariates = False

    def __init__(self, model_id: str, device: str = DEVICE, dtype: "Any" = DTYPE,
                 context_length: int = CONTEXT_LENGTH, prediction_length: int = HORIZON) -> None:
        self._ctx_len = context_length
        self._pred_len = prediction_length
        super().__init__(model_id, device, dtype)

    def load(self) -> None:
        from uni2ts.model.moirai2 import Moirai2Forecast, Moirai2Module

        self._module = Moirai2Module.from_pretrained(self.model_id)
        self._forecast_cls = Moirai2Forecast
        self._n_params = sum(p.numel() for p in self._module.parameters())

    def _build_model(self, context_length: int):
        return self._forecast_cls(
            module=self._module,
            prediction_length=self._pred_len,
            context_length=context_length,
            target_dim=1,
            feat_dynamic_real_dim=0,
            past_feat_dynamic_real_dim=0,
        )

    def predict(
        self,
        context: np.ndarray,
        h: int = HORIZON,
        num_samples: int = 100,
        quantiles: list[float] | None = None,
        covariates_past: np.ndarray | None = None,
        covariates_future: np.ndarray | None = None,
        context_length: int | None = None,
        **kwargs: Any,
    ) -> dict[str, np.ndarray]:
        if covariates_past is not None or covariates_future is not None:
            warnings.warn(f"[{self.name}] usa apenas dynamic real feats; ignorando covariáveis aqui.")
        quantiles = quantiles or QUANTILES
        cl = min(context_length or self._ctx_len, len(context))
        ctx = self._fit_context(context, cl)

        from gluonts.dataset.common import ListDataset

        ds = ListDataset(
            [{"target": ctx.astype(np.float32), "start": pd.Period("2000-01-01", freq="H")}],
            freq="H",
        )
        model = self._build_model(context_length=len(ctx))
        predictor = model.create_predictor(batch_size=1)
        with self._autocast():
            forecast = next(iter(predictor.predict(ds)))
        # forecast.samples: (num_samples, H) -> quantis empíricos
        samples = np.asarray(forecast.samples)[:, :h]
        q_np = np.stack([np.quantile(samples, q, axis=0) for q in quantiles], axis=0)
        median = np.quantile(samples, 0.5, axis=0)
        return {"median": median, "quantiles": q_np, "samples": samples}


In [ ]:
class AutoARIMAWrapper:
    """Baseline clássico AutoARIMA (statsforecast). Interface compatível com evaluate."""

    name = "autoarima"
    supports_covariates = False
    num_params = 0

    def __init__(self, season_length: int = 24) -> None:
        self.season_length = season_length

    def unload(self) -> None:
        free_memory()

    def predict(
        self,
        context: np.ndarray,
        h: int = HORIZON,
        num_samples: int = 100,
        quantiles: list[float] | None = None,
        covariates_past: np.ndarray | None = None,
        covariates_future: np.ndarray | None = None,
        context_length: int | None = None,
        **kwargs: Any,
    ) -> dict[str, np.ndarray]:
        quantiles = quantiles or QUANTILES
        from statsforecast import StatsForecast
        from statsforecast.models import AutoARIMA

        ctx = np.asarray(context, dtype=float)
        if context_length:
            ctx = ctx[-context_length:]
        tmp = pd.DataFrame(
            {
                "unique_id": "s",
                "ds": pd.date_range("2000-01-01", periods=len(ctx), freq="H"),
                "y": ctx,
            }
        )
        # níveis de IC simétricos para cobrir os quantis pedidos
        levels = sorted({int(round(abs(1 - 2 * q) * 100)) for q in quantiles if q != 0.5})
        levels = [l for l in levels if 0 < l < 100] or [80]
        sf = StatsForecast(models=[AutoARIMA(season_length=self.season_length)], freq="H")
        fc = sf.forecast(df=tmp, h=h, level=levels)

        mean = fc["AutoARIMA"].to_numpy()
        q_rows = []
        for q in quantiles:
            if q == 0.5:
                q_rows.append(mean)
                continue
            lvl = int(round(abs(1 - 2 * q) * 100))
            lvl = min(levels, key=lambda L: abs(L - lvl))
            col = f"AutoARIMA-lo-{lvl}" if q < 0.5 else f"AutoARIMA-hi-{lvl}"
            q_rows.append(fc[col].to_numpy() if col in fc.columns else mean)
        q_np = np.stack(q_rows, axis=0)
        return {"median": mean, "quantiles": q_np}


# Registro: nome -> (classe, kwargs). Instanciados sob demanda (um por vez).
WRAPPER_REGISTRY: dict[str, tuple[type | Any, dict[str, Any]]] = {
    "chronos-bolt-base": (ChronosBoltWrapper, {"model_id": MODEL_IDS["chronos-bolt-base"]}),
    "chronos-2": (Chronos2Wrapper, {"model_id": MODEL_IDS["chronos-2"]}),
    "timesfm-2.0": (TimesFMWrapper, {"model_id": MODEL_IDS["timesfm-2.0"]}),
    "moirai-2-small": (Moirai2Wrapper, {"model_id": MODEL_IDS["moirai-2-small"]}),
}
print("Modelos registrados:", list(WRAPPER_REGISTRY) + ["autoarima"])


## 5. Benchmark zero-shot

Para cada modelo: instancia (carrega), prevê na janela de teste de **todas** as séries,
mede latência e pico de VRAM, e então **descarrega** antes do próximo modelo.
Resultados consolidados em `results_zeroshot`. Falhas por modelo/série são capturadas
e não interrompem o benchmark.


In [ ]:
def detect_seasonality(freq: str) -> int:
    """Período sazonal padrão para MASE/AutoARIMA a partir da frequência."""
    table = {"H": 24, "D": 7, "W": 52, "M": 12, "MS": 12, "15min": 96, "30min": 48, "T": 1440}
    return table.get(freq, 1)


SEASONALITY = detect_seasonality(DATA_FREQ)


def run_zeroshot_for_model(
    model_key: str,
    test_windows: list[dict[str, Any]],
    context_length: int = CONTEXT_LENGTH,
    quantiles: list[float] | None = None,
) -> tuple[list[dict[str, Any]], dict[str, float]]:
    """Carrega um modelo, avalia em todas as janelas, descarrega. Retorna (linhas, meta)."""
    quantiles = quantiles or QUANTILES
    rows: list[dict[str, Any]] = []
    reset_gpu_peak()

    if model_key == "autoarima":
        wrapper: Any = AutoARIMAWrapper(season_length=SEASONALITY)
    else:
        cls, kw = WRAPPER_REGISTRY[model_key]
        wrapper = cls(**kw)
    log_gpu_mem(f"{model_key} carregado")

    latencies: list[float] = []
    for w in test_windows:
        ctx = w["context"]
        cov_past = w.get("cov_past")
        cov_fut = w.get("cov_future")
        try:
            t0 = time.perf_counter()
            fc = wrapper.predict(
                ctx, h=HORIZON, quantiles=quantiles, context_length=context_length,
                covariates_past=cov_past, covariates_future=cov_fut,
            )
            latencies.append((time.perf_counter() - t0) * 1000.0)
            metrics = evaluate(
                fc, w["target"], train_history=ctx,
                quantile_levels=quantiles, seasonality=SEASONALITY,
            )
            rows.append({"model": model_key, "unique_id": w["unique_id"], **metrics})
        except Exception as exc:
            warnings.warn(f"[{model_key}/{w['unique_id']}] previsão falhou: {exc}")
            rows.append({"model": model_key, "unique_id": w["unique_id"],
                         "MAE": np.nan, "RMSE": np.nan, "sMAPE": np.nan,
                         "WQL": np.nan, "CRPS": np.nan, "MASE": np.nan})

    peak = log_gpu_mem(f"{model_key} após inferência")
    meta = {
        "model": model_key,
        "num_params": getattr(wrapper, "num_params", 0),
        "latency_ms_mean": float(np.mean(latencies)) if latencies else float("nan"),
        "peak_vram_mb": peak,
    }
    wrapper.unload()
    del wrapper
    free_memory()
    return rows, meta


In [ ]:
# Split + janelas de teste (uma janela final por série, alinhada a HORIZON)
train_df, val_df, test_df = split_panel(df, h_test=HORIZON, h_val=HORIZON)

# Para o benchmark usamos a última janela de cada série: contexto = histórico até o teste.
def build_eval_windows(df_full: pd.DataFrame, context_length: int = CONTEXT_LENGTH) -> list[dict[str, Any]]:
    """Uma janela por série: últimos HORIZON pontos como alvo, anteriores como contexto."""
    wins: list[dict[str, Any]] = []
    cov_cols = [c for c in df_full.columns if c not in REQUIRED_COLS]
    for uid, g in df_full.groupby("unique_id"):
        g = g.sort_values("ds").reset_index(drop=True)
        if len(g) < HORIZON + 1:
            continue
        cut = len(g) - HORIZON
        ctx = g["y"].to_numpy(dtype=float)[:cut][-context_length:]
        tgt = g["y"].to_numpy(dtype=float)[cut:]
        win: dict[str, Any] = {"unique_id": uid, "context": ctx, "target": tgt,
                               "ds_target": g["ds"].to_numpy()[cut:]}
        if cov_cols:
            win["cov_past"] = g[cov_cols].to_numpy(dtype=float)[:cut][-context_length:]
            win["cov_future"] = g[cov_cols].to_numpy(dtype=float)[cut:]
        wins.append(win)
    return wins


eval_windows = build_eval_windows(df, CONTEXT_LENGTH)
print(f"{len(eval_windows)} janelas de avaliação (1 por série elegível).")


In [ ]:
# Loop principal zero-shot (carrega/usa/descarrega cada modelo)
MODELS_TO_RUN = ["chronos-bolt-base", "chronos-2", "timesfm-2.0", "moirai-2-small", "autoarima"]

all_rows: list[dict[str, Any]] = []
model_meta: list[dict[str, Any]] = []

for mk in MODELS_TO_RUN:
    print(f"\n===== Zero-shot: {mk} =====")
    try:
        rows, meta = run_zeroshot_for_model(mk, eval_windows, CONTEXT_LENGTH)
        all_rows.extend(rows)
        model_meta.append(meta)
    except Exception as exc:
        warnings.warn(f"Modelo {mk} indisponível/falhou no carregamento: {exc}. Pulando.")
        free_memory()

results_zeroshot = pd.DataFrame(all_rows)
meta_df = pd.DataFrame(model_meta).set_index("model") if model_meta else pd.DataFrame()
print("\nResumo zero-shot (média por modelo):")
results_zeroshot.groupby("model")[["MAE", "RMSE", "sMAPE", "MASE", "WQL", "CRPS"]].mean() if not results_zeroshot.empty else results_zeroshot


In [ ]:
# Tabela média por modelo
if not results_zeroshot.empty:
    zs_summary = results_zeroshot.groupby("model")[["MAE", "RMSE", "sMAPE", "MASE", "WQL", "CRPS"]].mean()
    display(zs_summary.style.background_gradient(cmap="RdYlGn_r", axis=0).format("{:.4f}"))
else:
    print("Nenhum resultado zero-shot — verifique a instalação dos modelos.")


In [ ]:
# Plot de previsões medianas + faixa [0.1, 0.9] para séries representativas.
def pick_representative_series(profile: pd.DataFrame, k: int = 3) -> list[str]:
    """Escolhe séries de perfis distintos: tendência forte, alta ACF (sazonal/suave), ruidosa."""
    if profile.empty:
        return []
    picks: list[str] = []
    by_trend = profile["mk_tau"].abs().sort_values(ascending=False)
    if len(by_trend):
        picks.append(by_trend.index[0])
    by_acf = profile["acf_lag1"].sort_values(ascending=False)
    for uid in by_acf.index:
        if uid not in picks:
            picks.append(uid)
            break
    by_noise = profile["acf_lag1"].sort_values(ascending=True)
    for uid in by_noise.index:
        if uid not in picks:
            picks.append(uid)
            break
    return picks[:k]


def plot_model_forecasts(series_ids: list[str], context_length: int = CONTEXT_LENGTH) -> None:
    """Recarrega cada modelo e plota previsões lado a lado para as séries escolhidas."""
    if not series_ids:
        print("Sem séries para plotar.")
        return
    wmap = {w["unique_id"]: w for w in eval_windows}
    series_ids = [s for s in series_ids if s in wmap]
    forecasts: dict[str, dict[str, dict[str, np.ndarray]]] = {s: {} for s in series_ids}

    for mk in MODELS_TO_RUN:
        try:
            if mk == "autoarima":
                wrapper: Any = AutoARIMAWrapper(season_length=SEASONALITY)
            else:
                cls, kw = WRAPPER_REGISTRY[mk]
                wrapper = cls(**kw)
        except Exception as exc:
            warnings.warn(f"[plot] {mk} indisponível: {exc}")
            continue
        for s in series_ids:
            try:
                forecasts[s][mk] = wrapper.predict(
                    wmap[s]["context"], h=HORIZON, context_length=context_length
                )
            except Exception as exc:
                warnings.warn(f"[plot] {mk}/{s} falhou: {exc}")
        wrapper.unload()
        del wrapper
        free_memory()

    n = len(series_ids)
    fig, axes = plt.subplots(n, 1, figsize=(12, 3.2 * n), squeeze=False)
    lo_i = QUANTILES.index(0.1)
    hi_i = QUANTILES.index(0.9)
    for ax, s in zip(axes.flat, series_ids):
        w = wmap[s]
        hist = w["context"][-min(len(w["context"]), 2 * HORIZON):]
        x_hist = np.arange(-len(hist), 0)
        x_fut = np.arange(0, HORIZON)
        ax.plot(x_hist, hist, color="black", lw=1.0, label="histórico")
        ax.plot(x_fut, w["target"], color="black", lw=1.5, ls="--", label="real")
        for mk, fc in forecasts[s].items():
            (line,) = ax.plot(x_fut, fc["median"], lw=1.2, label=mk)
            ax.fill_between(x_fut, fc["quantiles"][lo_i], fc["quantiles"][hi_i],
                            alpha=0.12, color=line.get_color())
        ax.axvline(0, color="grey", lw=0.6)
        ax.set_title(f"Série {s}", fontsize=10)
        ax.legend(fontsize=7, ncol=3)
    fig.suptitle("Previsões medianas + faixa [0.1, 0.9]", fontsize=12)
    fig.tight_layout()
    plt.show()


rep_series = pick_representative_series(panel_profile, k=3)
print("Séries representativas:", rep_series)
plot_model_forecasts(rep_series)


## 6. Ajuste de hiperparâmetros com Optuna

### Estudo A — Tuning de inferência zero-shot (minimiza WQL na validação)

Espaço de busca (parâmetros aplicados conforme a API real de cada modelo):

- `context_length` ∈ {128, 256, 512, 1024, 2048} — todos os modelos.
- `num_samples` ∈ {20, 50, 100, 200} — apenas modelos que amostram (Moirai-2).
- `temperature` ∈ [0.5, 1.5] — apenas Chronos sampling (Bolt/2 ignoram se forem de quantis).
- `top_k` ∈ {10, 50} — quando aplicável.

Sampler: `TPESampler(seed=SEED)` · Pruner: `MedianPruner(n_warmup_steps=5)` ·
Storage: `InMemoryStorage` · `N_TRIALS_INFERENCE` trials por modelo.

> **Estudo B (LoRA fine-tuning):** fora de escopo nesta versão (decisão do usuário).
> O esqueleto de constantes (`N_TRIALS_LORA`) permanece para retomada futura.


In [ ]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Janelas de validação (alvo = bloco de validação; contexto = histórico anterior a ele)
def build_val_windows(df_full: pd.DataFrame, max_context: int = 2048) -> list[dict[str, Any]]:
    """Uma janela de validação por série (alvo HORIZON antes do teste)."""
    wins: list[dict[str, Any]] = []
    for uid, g in df_full.groupby("unique_id"):
        g = g.sort_values("ds").reset_index(drop=True)
        if len(g) < 2 * HORIZON + 1:
            continue
        cut = len(g) - 2 * HORIZON  # deixa o último HORIZON para teste
        ctx = g["y"].to_numpy(dtype=float)[:cut][-max_context:]
        tgt = g["y"].to_numpy(dtype=float)[cut: cut + HORIZON]
        wins.append({"unique_id": uid, "context": ctx, "target": tgt})
    return wins


val_windows = build_val_windows(df)
print(f"{len(val_windows)} janelas de validação para o Optuna.")

# Quais hiperparâmetros cada modelo realmente usa
MODEL_SEARCH_FLAGS: dict[str, dict[str, bool]] = {
    "chronos-bolt-base": {"num_samples": False, "temperature": False, "top_k": False},
    "chronos-2":         {"num_samples": False, "temperature": False, "top_k": False},
    "timesfm-2.0":       {"num_samples": False, "temperature": False, "top_k": False},
    "moirai-2-small":    {"num_samples": True,  "temperature": False, "top_k": False},
}


In [ ]:
def make_objective(model_key: str, windows: list[dict[str, Any]]) -> Callable[[optuna.Trial], float]:
    """Cria a função objetivo do Optuna (Estudo A) para um modelo.

    Carrega o modelo UMA vez por trial, avalia WQL médio na validação e descarrega.
    Reporta a métrica de forma incremental para permitir pruning (MedianPruner).
    """
    flags = MODEL_SEARCH_FLAGS[model_key]

    def objective(trial: optuna.Trial) -> float:
        context_length = trial.suggest_categorical("context_length", [128, 256, 512, 1024, 2048])
        kw: dict[str, Any] = {"context_length": context_length}
        if flags["num_samples"]:
            kw["num_samples"] = trial.suggest_categorical("num_samples", [20, 50, 100, 200])
        if flags["temperature"]:
            kw["temperature"] = trial.suggest_float("temperature", 0.5, 1.5)
        if flags["top_k"]:
            kw["top_k"] = trial.suggest_categorical("top_k", [10, 50])

        cls, base_kw = WRAPPER_REGISTRY[model_key]
        wrapper = cls(**base_kw)
        wqls: list[float] = []
        try:
            for i, w in enumerate(windows):
                try:
                    fc = wrapper.predict(w["context"], h=HORIZON, quantiles=QUANTILES, **kw)
                    wqls.append(evaluate(fc, w["target"], quantile_levels=QUANTILES)["WQL"])
                except Exception:
                    wqls.append(float("nan"))
                # reporta média parcial para o pruner
                trial.report(float(np.nanmean(wqls)), step=i)
                if trial.should_prune():
                    raise optuna.TrialPruned()
        finally:
            wrapper.unload()
            del wrapper
            free_memory()
        score = float(np.nanmean(wqls)) if wqls else float("inf")
        return score if np.isfinite(score) else float("inf")

    return objective


def run_study_a(model_key: str, n_trials: int = N_TRIALS_INFERENCE) -> optuna.Study:
    """Executa o Estudo A para um modelo e retorna o Study."""
    study = optuna.create_study(
        direction="minimize",
        sampler=TPESampler(seed=SEED),
        pruner=MedianPruner(n_warmup_steps=5),
        storage=optuna.storages.InMemoryStorage(),
        study_name=f"inference-{model_key}",
    )
    study.optimize(make_objective(model_key, val_windows), n_trials=n_trials, show_progress_bar=False)
    return study


In [ ]:
# Executa o Estudo A para os modelos neurais disponíveis.
STUDY_A_MODELS = ["chronos-bolt-base", "chronos-2", "timesfm-2.0", "moirai-2-small"]

studies_a: dict[str, optuna.Study] = {}
for mk in STUDY_A_MODELS:
    print(f"\n===== Optuna Study A: {mk} ({N_TRIALS_INFERENCE} trials) =====")
    try:
        studies_a[mk] = run_study_a(mk, N_TRIALS_INFERENCE)
        best = studies_a[mk].best_trial
        print(f"  melhor WQL={best.value:.4f}  params={best.params}")
    except Exception as exc:
        warnings.warn(f"Study A falhou para {mk}: {exc}")
        free_memory()

print("\nEstudos concluídos:", list(studies_a))


In [ ]:
# trials_dataframe inline para cada estudo
for mk, st in studies_a.items():
    print(f"\n--- trials: {mk} ---")
    display(st.trials_dataframe().sort_values("value").head(10))


In [ ]:
# Plots Optuna inline (matplotlib): history, importances, parallel coordinate
import optuna.visualization.matplotlib as ovm

for mk, st in studies_a.items():
    completed = [t for t in st.trials if t.state == optuna.trial.TrialState.COMPLETE]
    if len(completed) < 2:
        print(f"[{mk}] poucos trials completos para plotar.")
        continue
    print(f"\n===== Análise Optuna: {mk} =====")
    try:
        ovm.plot_optimization_history(st)
        plt.title(f"Optimization history — {mk}")
        plt.tight_layout(); plt.show()
    except Exception as exc:
        warnings.warn(f"history plot falhou ({mk}): {exc}")
    try:
        ovm.plot_param_importances(st)
        plt.title(f"Param importances — {mk}")
        plt.tight_layout(); plt.show()
    except Exception as exc:
        warnings.warn(f"importances plot falhou ({mk}): {exc}")
    try:
        ovm.plot_parallel_coordinate(st)
        plt.title(f"Parallel coordinate — {mk}")
        plt.tight_layout(); plt.show()
    except Exception as exc:
        warnings.warn(f"parallel coordinate plot falhou ({mk}): {exc}")


## 7. Comparação final

Tabela consolidada cruzando: melhor **zero-shot** vs. melhor **tuning Optuna (Estudo A)**,
mais latência média (ms), pico de VRAM (MB) e número de parâmetros por modelo.
(Estudo B / LoRA fora de escopo nesta versão.)


In [ ]:
def reevaluate_with_best(model_key: str, study: optuna.Study) -> dict[str, float]:
    """Reavalia um modelo no conjunto de TESTE usando os melhores hparâmetros do Study A."""
    params = dict(study.best_trial.params)
    cls, base_kw = WRAPPER_REGISTRY[model_key]
    wrapper = cls(**base_kw)
    metrics_rows: list[dict[str, float]] = []
    try:
        for w in eval_windows:
            try:
                fc = wrapper.predict(w["context"], h=HORIZON, quantiles=QUANTILES, **params)
                metrics_rows.append(evaluate(fc, w["target"], train_history=w["context"],
                                             quantile_levels=QUANTILES, seasonality=SEASONALITY))
            except Exception:
                pass
    finally:
        wrapper.unload(); del wrapper; free_memory()
    if not metrics_rows:
        return {}
    mdf = pd.DataFrame(metrics_rows)
    return {f"tuned_{k}": float(mdf[k].mean()) for k in ["WQL", "CRPS", "MASE", "MAE"]}


# Métricas zero-shot médias por modelo
final_parts: list[dict[str, Any]] = []
zs_means = (
    results_zeroshot.groupby("model")[["WQL", "CRPS", "MASE", "MAE"]].mean()
    if not results_zeroshot.empty else pd.DataFrame()
)

for mk in MODELS_TO_RUN:
    row: dict[str, Any] = {"model": mk}
    if mk in zs_means.index:
        for k in ["WQL", "CRPS", "MASE", "MAE"]:
            row[f"zeroshot_{k}"] = float(zs_means.loc[mk, k])
    if mk in studies_a:
        try:
            row.update(reevaluate_with_best(mk, studies_a[mk]))
            row["best_params"] = studies_a[mk].best_trial.params
        except Exception as exc:
            warnings.warn(f"reavaliação tuned falhou para {mk}: {exc}")
    if not meta_df.empty and mk in meta_df.index:
        row["latency_ms"] = float(meta_df.loc[mk, "latency_ms_mean"])
        row["peak_vram_mb"] = float(meta_df.loc[mk, "peak_vram_mb"])
        row["num_params"] = int(meta_df.loc[mk, "num_params"])
    final_parts.append(row)

final_comparison = pd.DataFrame(final_parts).set_index("model")
final_comparison


In [ ]:
# Destaque do melhor modelo por métrica probabilística principal (WQL)
if not final_comparison.empty:
    cols = [c for c in ["zeroshot_WQL", "tuned_WQL", "zeroshot_CRPS", "tuned_CRPS",
                        "zeroshot_MASE", "latency_ms", "peak_vram_mb", "num_params"]
            if c in final_comparison.columns]
    view = final_comparison[cols].copy()
    num_cols = [c for c in cols if c != "num_params"]
    display(view.style.background_gradient(cmap="RdYlGn_r", subset=num_cols).format("{:.4f}", subset=num_cols))

    if "zeroshot_WQL" in final_comparison.columns:
        best_zs = final_comparison["zeroshot_WQL"].idxmin()
        print(f"Melhor zero-shot (WQL): {best_zs}")
    if "tuned_WQL" in final_comparison.columns and final_comparison["tuned_WQL"].notna().any():
        best_tuned = final_comparison["tuned_WQL"].idxmin()
        print(f"Melhor com tuning Optuna A (WQL): {best_tuned}")
else:
    print("Sem comparação final — nenhum modelo produziu resultados.")


## 8. Análise crítica

> Esta célula é um **roteiro de interpretação**. Após executar com seus dados,
> cruze os números de `final_comparison` com `panel_profile` (`summarize_panel`)
> para preencher as conclusões específicas do seu painel.

**Como ler os resultados em função do perfil das séries**

- **Séries sazonais / ACF lag-1 alta** (`acf_lag1` elevado): TSFMs treinados em larga escala
  (Chronos-2, TimesFM) tendem a capturar bem padrões periódicos sem ajuste. Compare `WQL`/`CRPS`
  desses modelos contra o AutoARIMA com `season_length` correto — frequentemente o ganho do TSFM
  encolhe quando o ARIMA "acerta" a sazonalidade.

- **Tendência forte** (`trend` = crescente/decrescente, `|mk_tau|` alto): modelos com contexto
  longo (aumentar `context_length` no Estudo A costuma ajudar) capturam melhor a deriva. Verifique
  se o melhor `context_length` encontrado pelo Optuna foi grande nessas séries.

- **Séries ruidosas** (`acf_lag1` baixa, `cv` alto): aqui a vantagem é da **incerteza calibrada**.
  Olhe `WQL`/`CRPS` (não só MAE): modelos probabilísticos como Chronos-Bolt/Chronos-2 e Moirai-2
  tendem a dominar o AutoARIMA por quantis melhor calibrados, mesmo com MAE parecido.

- **Séries curtas** (`length` pequena, < L): o truncamento adaptativo entra em ação. TSFMs zero-shot
  costumam degradar menos que o ARIMA (que precisa estimar parâmetros), mas confira os avisos de
  contexto curto emitidos durante a execução.

**Trade-offs custo × precisão**

- Cruze `latency_ms` e `peak_vram_mb` com o ganho de `WQL`. TimesFM-2.0 (~500M) é o mais pesado;
  Chronos-2 (~120M) e Moirai-2 small tendem a oferecer melhor relação custo/precisão em 12 GB.
- O Estudo A raramente muda a ordem dos modelos, mas costuma reduzir `WQL` ajustando `context_length`
  — ganho barato (sem treino). Avalie se o ganho marginal justifica o tempo de tuning.

**Quando vale (ou não) TSFM vs. AutoARIMA**

- **Vale TSFM:** muitas séries heterogêneas, necessidade de previsão probabilística calibrada,
  pouca/nenhuma série para treinar modelo clássico por série, ou presença de covariáveis (→ Chronos-2).
- **Vale AutoARIMA:** poucas séries bem comportadas, sazonalidade clara e estável, restrição de
  GPU/latência, ou exigência de modelo interpretável. Se o ARIMA empata em `WQL` com custo muito menor,
  ele é a escolha pragmática.

*(Preencha abaixo as conclusões específicas do seu painel após a execução.)*
